Эксперимент посвящен определению лучшего способа построение графа для документов. 

Идея в том, что связи, которые должны быть на самом деле могут не присутствовать на самом деле (из-за) построения графа. 
В этом смысле верхняя грань это полносвязный граф. Другой вопрос, а как сделать "оптимально" и что под этим понимать.

# Необходимые импорты и классы

In [1]:
import sys
import os
import datetime
from dotenv import load_dotenv
sys.path.append('../..')
env_file = os.path.join('../..', '.env')
load_dotenv(env_file)
dataset_path = os.environ['DATASET_PATH']
test_path = os.environ['TEST_PATH']
test_coco_path = os.environ['TEST_COCO_PATH']
coco_path = os.environ['COCO_PATH']
BASE_PATH = os.getcwd()
# cache_pdf = os.environ['CASH_PDF_PATH']

In [2]:
from rows2regionsGLAM.utils.pdf_manager import PDFManager
from rows2regionsGLAM.utils.loger import Loger
from rows2regionsGLAM.utils.row_manager import RowManager
from rows2regionsGLAM.utils.ploter import Ploter
from rows2regionsGLAM.tokenizers import RowGLAMTokenizer
from rows2regionsGLAM.utils.coco_manager import COCOManager
from rows2regionsGLAM.datasetloaders.base_line_dataset import GLAMDataset
from rows2regionsGLAM.utils.cacher import Cacher
from rows2regionsGLAM.converters import Rows2Regions
from rows2regionsGLAM.utils.tester import Tester as BaseTester

/home/daniil/project/PageR/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
loger = Loger(f'log_{datetime.date.today()}.txt')
pdf_manager = PDFManager(conf={"loger": loger, "pdf_reader": "PDFMiner"})
row_manager = RowManager(conf={"loger": loger, "add_image": True})
coco_manager = COCOManager(conf={"loger": loger, "coco_path": coco_path})
ploter = Ploter(conf={"loger": loger})


In [4]:
# Метрики для оценки
from rows2regionsGLAM.metrics import GridMetric
from torchmetrics.detection.mean_ap import MeanAveragePrecision

In [5]:
from pager.page_model.sub_models import RegionModel, RowsModel

rows_model = RowsModel()
region_model = RegionModel()


In [6]:
CLASSES = {1: 'text', 2: 'title', 3: 'list', 4: 'table', 5: 'figure', 0: 'other'}

In [7]:
class TrueModel:
    def __call__(self, data_graph_dict):
        return {
            "node_classes": data_graph_dict["true_nodes"],
            "E_pred": data_graph_dict["true_edges"] 
        }

class Tester(BaseTester):
    def calculate_target_and_preds_with_json(self, test_dataset, name_dataset, name_test_dataset, dataset_path, test_path, fun_pred_region):
        target = []
        preds = []
        word_grids = []
        row_grids = []
        N = len(test_dataset)
        for i, d in enumerate(test_dataset):
            name_file = test_dataset.pdf_names[i]
            true_regions = test_dataset.coco_ann[name_file]['regions']
            bboxes_true = self.clean_bboxes_true([reg['segment'] for reg in true_regions])
            
            pdf_json, pdf_img = self.pdf_manager.get_json_and_img_from_pdf(os.path.join(test_path, name_file))
            w, h = pdf_json['width'],pdf_json['height']
            if name_test_dataset == "doclaynet":
                resize = (w/1024, h/1024)
            else:
                resize = (1, 1)
            row_json = self.row_manager.get_row_json_from_pdf_json(pdf_json)
            # self.clean_rows(row_json, bboxes_true)
            
            # try:
            pred_regions = fun_pred_region(self.rows2regions, row_json, d)

            if name_dataset == "doclaynet" and name_test_dataset == "publaynet":
                for r in pred_regions:
                    pub_label = DOC2PUB_MAP.get(r['label'], 'other')
                    r['label'] = pub_label
                # pred_regions = aggregate_list_items(pred_regions)

            bboxes_pred = [r['segment'] for r in pred_regions if r['label'] != 'other']

            word_grids.append([self.get_bbox(word['segment']) for row in row_json for word in row['words']])
            row_grids.append([self.get_bbox(row['segment']) for row in row_json])
            target.append([self.get_bbox(seg, resize) for seg in bboxes_true])
            preds.append([self.get_bbox(seg) for seg in bboxes_pred])
            # except:
            #     print(i,d["file_name"])

            
            print(f"{(i+1)/N*100:4.2f} %", end='\r')

        return [target, preds, word_grids, row_grids]

model = TrueModel()




# 0. Базовая функция для эксперимента

In [10]:
import numpy as np
from pager import ImageSegment, Region
def exp(cache_pdf, graph_creat):
    class ExpTokenizer(RowGLAMTokenizer):
        def get_A(self, rows_json):
            
            edges = graph_creat([ImageSegment(dict_2p=row_json['segment']) for row_json in rows_json])
    
            A1, A2 = [], []
            for a1, a2 in edges:
                A1.append(a1)
                A2.append(a2)
            index = np.argsort(A1)
            A1_ = [A1[i] for i in index]
            A2_ = [A2[i] for i in index]
        
            return [A1_, A2_]
    
    tokenizer = ExpTokenizer() 
    
    rows2regions = Rows2Regions({
        'model': model, 
        'tokenizer': tokenizer,
        'is_merge_extract': True,
        'classes': CLASSES
    })
    
    pdf2torch_dict = Cacher({
        "loger": loger,
        "pdf_manager": pdf_manager,
        "row_manager": row_manager,
        "tokenizer": tokenizer
    })
    
    test_dataset = GLAMDataset(
        {
        "name_dataset": "publaynet",
        "loger": loger,
        "pdf_dir": test_path,
        "coco_file": test_coco_path,
        "count_class": 6,
        "default_index": 0,
        "cache_dir": cache_pdf,
        "pdf2torch_dict": pdf2torch_dict
        }
    )
    
    def fun_pred_region(rows2regions, rows_json, graph_dict_torch):
        result = rows2regions.rows2regionsGLAM(graph_dict_torch)
        result['deleted_edges'] = result['E_pred'] < 0.5
        
        graph = graph_dict_torch['inds']
        deleted_edges = result['deleted_edges']
        node_classes = result['node_classes']
        regions = rows2regions.regions_from_graph(rows_json, graph, deleted_edges, node_classes)
        return [Region(r).to_dict() for r in regions ]

    
    for i, d in enumerate(test_dataset):
        print(f'count: {i}', end='\r')


    tester = Tester(conf={
        "loger": loger, 
        "pdf_manager": pdf_manager, 
        "row_manager": row_manager, 
        "rows_model": rows_model, 
        "region_model": region_model, 
        "rows2regions": rows2regions})
    metrics = tester.calculate_target_and_preds_with_json(
        test_dataset=test_dataset, 
        name_dataset='publaynet', 
        name_test_dataset='publaynet', 
        dataset_path=dataset_path, 
        test_path=test_path,
        fun_pred_region=fun_pred_region
    )
    tester.print_result(metrics)
    

# 1. Полносвязный граф

In [11]:
# Путь для хранения 
cache_pdf = os.path.join(BASE_PATH, 'tmp_test_cache_full')

def graph_creat(segments):
    N = len(segments)
    return [(i, j) for i in range(N) for j in range(i, N)]
    
exp(cache_pdf, graph_creat)




Cannot set gray non-stroke color because /'P1' is an invalid float value
Cannot set gray non-stroke color because /'P2' is an invalid float value
Cannot set gray non-stroke color because /'P3' is an invalid float value
Cannot set gray non-stroke color because /'P4' is an invalid float value


Cannot set gray non-stroke color because /'P5' is an invalid float value
Cannot set gray non-stroke color because /'P6' is an invalid float value
Cannot set gray non-stroke color because /'P7' is an invalid float value
Cannot set gray non-stroke color because /'P8' is an invalid float value


Cannot set gray non-stroke color because /'P1' is an invalid float value
Cannot set gray non-stroke color because /'P2' is an invalid float value
Cannot set gray non-stroke color because /'P3' is an invalid float value
Cannot set gray non-stroke color because /'P3' is an invalid float value
Cannot set gray non-stroke color because /'P4' is an invalid float value
Cannot set gray non-stroke color because /'P4' is an invalid float value
Cannot set gray non-stroke color because /'P5' is an invalid float value
Cannot set gray non-stroke color because /'P5' is an invalid float value


mAP@IoU[0.50:0.95]   :0.60767400
threshold_05--------------------
precision_row       :0.9894
recall_row          :0.9616
f1_row              :0.9753
precision_word      :0.9478
recall_word         :0.9216
f1_word             :0.9345
threshold_95--------------------
precision_row       :0.9237
recall_row          :0.8997
f1_row              :0.9116
precision_word      :0.8880
recall_word         :0.8655
f1_word             :0.8766



# Манхетовское расстояние 

In [12]:
cache_pdf = os.path.join(BASE_PATH, 'tmp_test_cache')

def graph_creat(segments):
    def fun_dist_bottom(seg1: ImageSegment, seg: ImageSegment):
        DIST = 3
        r1 = seg1.x_bottom_right
        r = seg.x_bottom_right
        l1 = seg1.x_top_left
        l = seg.x_top_left

        x1c, y1c = seg1.get_center()
        xc, yc = seg.get_center()
        if y1c > yc: # Только в одном направление
            return np.inf
        
        if abs(x1c-xc)+abs(y1c-yc) < DIST: # Если совпали
            return np.inf
        
        xd = min(abs(r1-r), abs(l1-l), abs(xc-x1c))
        yd = abs(y1c-yc) 
        
        if abs(r1-r) < DIST or abs(l1-l) < DIST or abs(xc-x1c) < DIST :
            return yd

        
        return xd+yd

    def fun_dist_right(seg1: ImageSegment, seg: ImageSegment):
        DIST = 3
        r1 = seg1.x_bottom_right
        r = seg.x_bottom_right
        l1 = seg1.x_top_left
        l = seg.x_top_left

        x1c, y1c = seg1.get_center()
        xc, yc = seg.get_center()
        if x1c > xc: # Только в одном направление
            return np.inf

        
        if abs(x1c-xc)+abs(y1c-yc) < DIST: # Если совпали
            return np.inf
        
        xd = min(abs(r1-l), abs(l1-r))
        yd = abs(y1c-yc) 

        h = (seg.height + seg1.height)/2
        if yd > 2*h:
            return np.inf
            
        
        return xd+yd

    dists_bottom = []
    for j, seg1 in enumerate(segments):
        dist_bottom = [fun_dist_bottom(seg1, seg) for seg in segments]
        if min(dist_bottom) == np.inf:
            continue
        k = int(np.argmin(dist_bottom))
        dists_bottom.append((min(j, k), max(j, k)))

    # dists_top = [(k, j) for j, k in dists_bottom]

    dists_right = []
    for j, seg1 in enumerate(segments):
        dist_right = [fun_dist_right(seg1, seg) for seg in segments]
        if min(dist_right) == np.inf:
            continue
        k = int(np.argmin(dist_right))
        dists_right.append((min(j, k), max(j, k)))

    # dists_left = [(k, j) for j, k in dists_right]

    all_edges = dists_bottom + dists_right
    all_edges = list(set(all_edges))
    return all_edges
    
exp(cache_pdf, graph_creat)

Cannot set gray non-stroke color because /'P1' is an invalid float value
Cannot set gray non-stroke color because /'P2' is an invalid float value
Cannot set gray non-stroke color because /'P3' is an invalid float value
Cannot set gray non-stroke color because /'P4' is an invalid float value


Cannot set gray non-stroke color because /'P5' is an invalid float value
Cannot set gray non-stroke color because /'P6' is an invalid float value
Cannot set gray non-stroke color because /'P7' is an invalid float value
Cannot set gray non-stroke color because /'P8' is an invalid float value


Cannot set gray non-stroke color because /'P1' is an invalid float value
Cannot set gray non-stroke color because /'P2' is an invalid float value


Cannot set gray non-stroke color because /'P3' is an invalid float value
Cannot set gray non-stroke color because /'P3' is an invalid float value
Cannot set gray non-stroke color because /'P4' is an invalid float value
Cannot set gray non-stroke color because /'P4' is an invalid float value
Cannot set gray non-stroke color because /'P5' is an invalid float value
Cannot set gray non-stroke color because /'P5' is an invalid float value


mAP@IoU[0.50:0.95]   :0.59622669
threshold_05--------------------
precision_row       :0.9648
recall_row          :0.9570
f1_row              :0.9609
precision_word      :0.9236
recall_word         :0.9173
f1_word             :0.9204
threshold_95--------------------
precision_row       :0.8955
recall_row          :0.8897
f1_row              :0.8926
precision_word      :0.8598
recall_word         :0.8554
f1_word             :0.8576

